# Wildfire Detection & Burn Severity
**NB 16** | California, USA | Sentinel-2 | dNBR USFS 4-class

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)


In [ ]:
BBOX        = (-122.50, 38.50, -122.00, 39.00)   # Sonoma County, California
DATE_BEFORE = ('2023-07-01', '2023-08-15')        # Pre-fire
DATE_AFTER  = ('2023-09-01', '2023-10-31')        # Post-fire
PROVIDERS   = ['planetary_computer']
BANDS       = ['B04','B08','B12']                 # Red, NIR, SWIR2 for NBR
DATA_DIR    = pathlib.Path('./results/nb16')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
pre_r  = client.search(bbox=BBOX, date_range=DATE_BEFORE, providers=PROVIDERS, cloud_cover_max=20)
post_r = client.search(bbox=BBOX, date_range=DATE_AFTER,  providers=PROVIDERS, cloud_cover_max=20)
print(f"Pre-fire: {len(pre_r)} scenes | Post-fire: {len(post_r)} scenes")
for label, r in [('PRE', pre_r), ('POST', post_r)]:
    if r: print(f"  {label}: {r[0].datetime[:10]}  cloud={r[0].cloud_cover:.1f}%")


In [ ]:
PRE_DIR  = DATA_DIR/'pre';  PRE_DIR.mkdir(exist_ok=True)
POST_DIR = DATA_DIR/'post'; POST_DIR.mkdir(exist_ok=True)
pre_dl  = client.download(pre_r[:1],  output_dir=str(PRE_DIR),  bands=BANDS, post_process=['reproject:EPSG:32610','cog'])
post_dl = client.download(post_r[:1], output_dir=str(POST_DIR), bands=BANDS, post_process=['reproject:EPSG:32610','cog'])
pre_scene  = pre_dl[0].path  if pre_dl  and pre_dl[0].success  else None
post_scene = post_dl[0].path if post_dl and post_dl[0].success else None


In [ ]:
PRE_READY  = DATA_DIR/'pre_ready.tif'; POST_READY = DATA_DIR/'post_ready.tif'
for scene, out, label in [(pre_scene, PRE_READY,'PRE'), (post_scene, POST_READY,'POST')]:
    if scene:
        scl = next(iter(scene.parent.rglob('*SCL*.tif')), None)
        rep = client.validator.validate(str(scene), required_bands=len(BANDS))
        r   = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=BBOX,
            scl_path=str(scl) if scl else None, normalise='scale_factor',
            scale_factor=10000.0, model_type='segmentation', output_path=str(out))
        print(f"{label}: {r['steps']}  shape={r['shape']}")


In [ ]:
# dNBR = NBR_pre - NBR_post (USFS protocol)
# Band map: Red=1, NIR=2, SWIR2=3
import rasterio

if PRE_READY.exists() and POST_READY.exists():
    idx_dir = DATA_DIR/'indices'; idx_dir.mkdir(exist_ok=True)
    nbr_pre  = client.indices.nbr(str(PRE_READY),  nir_band=2, swir2_band=3, output_path=str(idx_dir/'nbr_pre.tif'))
    nbr_post = client.indices.nbr(str(POST_READY), nir_band=2, swir2_band=3, output_path=str(idx_dir/'nbr_post.tif'))
    with rasterio.open(nbr_pre) as s1, rasterio.open(nbr_post) as s2:
        dnbr = s1.read(1).astype(np.float32) - s2.read(1).astype(np.float32)
        prof = s1.profile.copy()
        res_m = abs(s1.transform.a)
    prof.update(dtype='float32', nodata=-9999, compress='lzw')
    DNBR_PATH = idx_dir / 'dnbr.tif'
    with rasterio.open(str(DNBR_PATH), 'w', **prof) as dst: dst.write(dnbr[np.newaxis])

    # USFS burn severity classification
    severity = np.zeros_like(dnbr, dtype=np.uint8)
    severity[(dnbr >= 0.10) & (dnbr < 0.27)] = 1  # Low severity
    severity[(dnbr >= 0.27) & (dnbr < 0.44)] = 2  # Moderate severity
    severity[dnbr >= 0.44]                    = 3  # High severity
    SEVERITY_PATH = DATA_DIR / 'burn_severity.tif'
    PREDICTION    = SEVERITY_PATH
    prof.update(dtype='uint8', nodata=255)
    with rasterio.open(str(SEVERITY_PATH), 'w', **prof) as dst: dst.write(severity[np.newaxis])

    CLASS_NAMES = {0:'Unburned', 1:'Low severity', 2:'Moderate severity', 3:'High severity'}
    for cls, name in CLASS_NAMES.items():
        area_ha = (severity == cls).sum() * res_m**2 / 10000
        print(f"  {name:<20}: {area_ha:>8.1f} ha  ({(severity==cls).mean()*100:.1f}%)")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=10, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=3, min_area_m2=50.0, simplify_tolerance=0.5)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB16 — Wildfire** complete.
- Study area: California USA
- Sensor: Sentinel-2 bi-temporal
- Model: dNBR USFS 4-class severity
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG